# 03 — Keyword Analysis

Separa Author Keywords e Index Keywords, calcula frecuencias, variantes y redes de coocurrencia.

> Notebook reproducible del Paper Semantic Bibliometric Toolkit v1.0.0. No elimina documentos ni convierte resultados exploratorios en conclusiones sin revisión manual.

## 1. Cargar corpus y reglas

In [ ]:

from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = None
current = Path.cwd().resolve()
for candidate in [current] + list(current.parents):
    if (candidate / "config.json").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró config.json.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from core import (
    find_project_root, read_json, load_config, ensure_directories,
    load_scopus_csv, load_clean_corpus, normalize_spacing,
    normalize_identifier, normalize_title, normalize_keyword,
    split_semicolon, frequency_table, save_csv, save_json,
    markdown_table, combined_text
)

CONFIG = load_config(PROJECT_ROOT)
ensure_directories(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = PROJECT_ROOT / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
THESAURUS_DIR = PROJECT_ROOT / "thesaurus"
VALIDATION_DIR = PROJECT_ROOT / "validation"

print("Project root:", PROJECT_ROOT)
print("Toolkit version:", CONFIG.get("toolkit_version"))
print("Configured corpus:", PROJECT_ROOT / CONFIG["input_csv"])

df = load_clean_corpus(PROJECT_ROOT)
from text_analysis import explode_keywords, document_frequency, keyword_variant_audit, keyword_cooccurrence
rules = read_json(PROJECT_ROOT / "config" / "keyword_rules.json")


## 2. Separar y normalizar keywords

In [ ]:

corrections = {}
corrections.update(rules.get("lexical_corrections", {}))
corrections.update(rules.get("known_merges", {}))
author_kw = explode_keywords(df, "Author Keywords", "Author Keyword", corrections)
index_kw = explode_keywords(df, "Index Keywords", "Index Keyword", corrections)
save_csv(author_kw, OUTPUT_DIR / "03_author_keywords_exploded.csv")
save_csv(index_kw, OUTPUT_DIR / "03_index_keywords_exploded.csv")
print("Author keyword occurrences:", len(author_kw))
print("Index keyword occurrences:", len(index_kw))


## 3. Frecuencias, cobertura y variantes

In [ ]:

author_frequency = document_frequency(author_kw)
index_frequency = document_frequency(index_kw)
author_variants = keyword_variant_audit(author_kw)
index_variants = keyword_variant_audit(index_kw)
coverage = pd.DataFrame([
    {"Source": "Author Keywords", "Documents with keywords": int(df["Author Keywords"].fillna("").astype(str).str.strip().ne("").sum())},
    {"Source": "Index Keywords", "Documents with keywords": int(df["Index Keywords"].fillna("").astype(str).str.strip().ne("").sum())}
])
coverage["Coverage (%)"] = (coverage["Documents with keywords"] / len(df) * 100).round(2)
save_csv(author_frequency, OUTPUT_DIR / "03_author_keyword_frequency.csv")
save_csv(index_frequency, OUTPUT_DIR / "03_index_keyword_frequency.csv")
save_csv(author_variants, OUTPUT_DIR / "03_author_keyword_variants.csv")
save_csv(index_variants, OUTPUT_DIR / "03_index_keyword_variants.csv")
save_csv(coverage, OUTPUT_DIR / "03_keyword_coverage.csv")
display(author_frequency.head(30))


## 4. Redes de coocurrencia preliminares

In [ ]:

thresholds = CONFIG.get("keyword_thresholds_to_export", [2, 3, 4, 5])
for threshold in thresholds:
    nodes_full, edges_full = keyword_cooccurrence(author_kw, int(threshold), fractional=False)
    nodes_fractional, edges_fractional = keyword_cooccurrence(author_kw, int(threshold), fractional=True)
    save_csv(nodes_full, OUTPUT_DIR / ("03_keyword_nodes_min" + str(threshold) + ".csv"))
    save_csv(edges_full, OUTPUT_DIR / ("03_keyword_edges_full_min" + str(threshold) + ".csv"))
    save_csv(edges_fractional, OUTPUT_DIR / ("03_keyword_edges_fractional_min" + str(threshold) + ".csv"))
print("Co-occurrence files exported for thresholds:", thresholds)


## 5. Resumen metodológico

In [ ]:

summary = {
    "Documents": len(df), "Unique Author Keywords": int(author_kw["Keyword normalized"].nunique()),
    "Unique Index Keywords": int(index_kw["Keyword normalized"].nunique()),
    "Author keyword coverage (%)": float(coverage.loc[coverage["Source"] == "Author Keywords", "Coverage (%)"].iloc[0]),
    "Index keyword coverage (%)": float(coverage.loc[coverage["Source"] == "Index Keywords", "Coverage (%)"].iloc[0]),
    "Top Author Keywords": author_frequency.head(20).to_dict("records")
}
save_json(summary, OUTPUT_DIR / "03_keyword_analysis_summary.json")
text = (
    "Author and indexed keywords were analyzed separately. Terms were normalized conservatively and counted by unique documents. "
    "Co-occurrence edge lists were generated under full and fractional counting for multiple occurrence thresholds."
)
(REPORTS_DIR / "03_keyword_methodology.txt").write_text(text, encoding="utf-8")
print(text)
